In [ ]:
%reload_ext autoreload
%autoreload 2
%matplotlib inline
from collections import defaultdict
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import seaborn as sns
import pandas as pd
from pathlib import Path
import tbparse

EXPERIMENT_NAME = "compare_methods_glue_roberta"
EXPERIMENT_DIR = Path("..") / "output" / EXPERIMENT_NAME
PLOT_DIR = Path("..") / "plots"
PLOT_DIR.mkdir(exist_ok=True)
PLOT_SUFFIX = ".pdf"

TASKS = ["mnli", "qnli", "sst2", "cola", "stsb", "qqp"]
METRICS = {
    "mnli": ["eval/accuracy", "eval/loss", "train/loss"],
    "sst2": ["eval/accuracy", "eval/loss", "train/loss"],
    "qqp": ["eval/accuracy", "eval/loss", "train/loss"],
    "qnli": ["eval/accuracy", "eval/loss", "train/loss"],
    "cola": ["eval/matthews_correlation", "eval/loss", "train/loss"],
    "stsb": ["eval/spearmanr", "eval/loss", "train/loss"],
}

In [ ]:
def get_task_dfs(dedup="last"):
    task_dfs = defaultdict(list)
    for file in EXPERIMENT_DIR.glob("*"):
        run_dir = Path(file)
        if not run_dir.is_dir():
            continue  # skip non-directories
        if run_dir.name.startswith("."):
            continue  # skip hidden directories
        print(run_dir)

        df = tbparse.SummaryReader(run_dir, pivot=False).scalars
        if dedup == "first":
            df = df.drop_duplicates(subset=["step", "tag"], keep="first")
        elif dedup == "last":
            df = df.drop_duplicates(subset=["step", "tag"], keep="last")
        elif dedup == "mean":
            df = df.groupby(["step", "tag"], as_index=False)["value"].mean()
        elif dedup == "max":
            df = df.groupby(["step", "tag"], as_index=False)["value"].max()
        else:
            raise ValueError(f"Unknown dedup method: {dedup}")

        df = df.pivot(index="step", columns="tag", values="value").reset_index()

        keyvals = file.name.split(",")
        df["Seed"] = int(keyvals[0].replace("seed=", ""))
        # df["Task"] = keyvals[1].replace("task=", "")
        task = keyvals[1].replace("task=", "")
        method = keyvals[2].replace("method=", "")
        df["Method"] = method
        rank = float(keyvals[3].replace("rank=", ""))
        # df["Rank"] = float(keyvals[3].replace("rank=", ""))
        lr = float(keyvals[4].replace("lr=", ""))
        df["LR"] = lr
        
        if method == "oplora_scaled" and lr != 0.2:
            continue

        task_dfs[(task, rank)].append(df)

    return {task: pd.concat(dfs_list).reset_index() for task, dfs_list in task_dfs.items()}

task_df_dict = get_task_dfs()

In [ ]:
display(next(iter(task_df_dict.values())).columns.tolist())
for (task, rank), df in task_df_dict.items():
    print(f"Task: {task}, Rank: {rank}")

In [ ]:
eval_df_dict = {}
best_lrs_dict = {}

for (task, rank), df in task_df_dict.items():
    eval_metrics = [col for col in df.columns if col.startswith("eval/")]
    main_metric = METRICS.get(task, ["eval/accuracy"])[0]
    if not eval_metrics:
        continue
    # Get df of eval metrics
    eval_df = df[["step", "Method", "LR"] + eval_metrics].dropna()
    # Average over seeds
    eval_df = eval_df.groupby(["step", "Method", "LR"], as_index=False).mean()
    # Get the best (averaged) eval accuracy over steps for each setting
    eval_df = eval_df.loc[eval_df.groupby(["Method", "LR"])[main_metric].idxmax()].reset_index(drop=True)
    # Get the best lr for each method by the max "best (averaged) eval accuracy"
    best_rows = (
        eval_df
        .groupby("Method")[["LR", main_metric]]
        .apply(lambda x: x.loc[x[main_metric].idxmax()])
        .sort_values(by=main_metric, ascending=False)
        .reset_index()
    )
    eval_df_dict[(task, rank)] = eval_df
    best_lrs_dict[(task, rank)] = best_rows


In [ ]:
# TASKS = ["mnli", "sst2", "qnli", "cola", "stsb"]
for task in TASKS:
    for rank in [8]:
        print(f"Task: {task}, Rank: {rank}")
        display(best_lrs_dict[(task, rank)])
        # display(eval_df_dict[(task, rank)])

In [ ]:
method_to_pretty = {
    "full": "Full",
    "lora": "LoRA",
    "svdlora": "SVDLoRA",
    "precond_lora": "RPLoRA",
    "oplora_proj": "Proj. PSI-LoRA",
    "oplora_scaled": "Scaled PSI-LoRA",
}

In [ ]:
y_min = {
    "mnli": 0.8,
    "qnli": 0.8,
    "sst2": 0.9,
    "cola": 0.45,
    "stsb": 0.86,
    "qqp": 0.8,
}

# --- Plot styling knobs (tweak these) ---
# Using sd can create very wide bands that dominate the figure.
# Good alternatives: "se" (standard error) or ("pi", 50) for a 50% prediction interval.
ERRORBAR = ("pi", 50)  # try: "se", ("ci", 68), None
ERR_ALPHA = 0.10         # transparency of the band
LINE_WIDTH = 2.2         # mean line thickness
GRID_ALPHA = 0.35

ranks = [8]
tasks = TASKS
fig, ax = plt.subplots(
     len(ranks), len(tasks),
     figsize=(2 + 3 * len(tasks), 2 + 3 * len(ranks)),
     squeeze=False
)

legend_handles = None
legend_labels = None

for i, rank in enumerate(ranks):
    for j, task in enumerate(tasks):
        plot_df = task_df_dict[(task, rank)].copy()
        metric_name = METRICS[task][0]

        # For each task method rank triple, select only the best lr
        plot_df = plot_df.merge(
            best_lrs_dict[(task, rank)][["Method", "LR"]],
            on=["Method", "LR"],
            how="inner",
        )

        plot_df["Method ($\\eta$)"] = plot_df.apply(
            lambda row: f"{method_to_pretty[row['Method']]} ($\\eta$={row['LR']})",
            axis=1
        )

        ax_ij = ax[i, j]
        ax_ij.grid(True, which="both", linestyle="--", linewidth=0.6, alpha=GRID_ALPHA)
        ax_ij.set_title(f"{task.upper()}")
        ax_ij.set_ylabel(metric_name)
        ax_ij.set_xlabel("Steps")
        # In this df, each method must have only one LR (the best one)
        lr_by_method = plot_df.groupby("Method")["LR"]
        assert lr_by_method.unique().apply(len).max() == 1, (
            "Each method should have only one LR in the filtered df"
        )
        method_lrs = lr_by_method.first().to_dict()
        # hue order should be ordered according to method_to_pretty, augmented with the lr
        hue_order = [
            f"{method_to_pretty[method]} ($\\eta$={method_lrs[method] if method in method_lrs else 'NA'})"
            for method in method_to_pretty.keys()
        ]

        # Only let seaborn build a legend once; then we'll place it globally.
        build_legend = (legend_handles is None)
        sns.lineplot(
            ax=ax_ij,
            data=plot_df,
            x="step",
            y=metric_name,
            hue="Method ($\\eta$)",
            hue_order=hue_order,
            palette="tab10",
            errorbar=ERRORBAR,
            err_style="band",
            err_kws={"alpha": ERR_ALPHA},
            linewidth=LINE_WIDTH,
            legend=build_legend,
        )

        if build_legend:
            legend_handles, legend_labels = ax_ij.get_legend_handles_labels()
            # Remove the axis-level legend; we'll render a single figure-level legend below.
            if ax_ij.legend_ is not None:
                ax_ij.legend_.remove()

        ax_ij.set_ylim(bottom=y_min[task] - 0.01, top=plot_df[metric_name].max() + 0.005)

# Place one global legend below the full figure in a single row
if legend_handles is not None and legend_labels is not None:
    fig.legend(
        legend_handles,
        legend_labels,
        fontsize=13,
        title="Method ($\\eta$)",
        loc="lower center",
        bbox_to_anchor=(0.5, -0.08),
        ncol=len(legend_labels),
        frameon=False,
    )
    fig.subplots_adjust(bottom=0.20)

plt.suptitle("RoBERTa-base on GLUE Tasks")
fig.tight_layout()

plt.savefig(PLOT_DIR / (EXPERIMENT_NAME + PLOT_SUFFIX), bbox_inches="tight")

plt.show()

In [ ]:
import numpy as np

def _format_mean_std(
    mean: float,
    std: float,
    *,
    digits: int = 2,
    scale: float = 100.0,
    pm: str = r"$\pm$",
    std_size_cmd: str = r"\scriptsize",
) -> str:
    """Format as `mean ± std` with optional scaling and small std."""
    if pd.isna(mean):
        return ""

    mean = float(mean) * scale
    if pd.isna(std):
        return f"{mean:.{digits}f}"

    std = float(std) * scale
    # Example: 87.70{\scriptsize $\pm$ 0.20}
    return f"{mean:.{digits}f}" + "{" + f"{std_size_cmd} {pm} {std:.{digits}f}" + "}"


def _latex_wrap(cmd: str, content: str) -> str:
    return cmd + "{" + content + "}"


def print_latex_table_best_metrics(
    *,
    task_df_dict: dict,
    best_lrs_dict: dict,
    tasks: list,
    rank: int,
    metrics_by_task: dict,
    method_to_pretty: dict | None = None,
    digits: int = 2,
    scale: float = 100.0,
    bold_best: bool = True,
    highlight_second_best: bool = True,
    second_best_cmd: str = r"\underline",  # e.g. "\underline" or "\textit"
    best_over: str = "steps",  # "steps" (max over steps per seed) or "fixed_step" (best mean step)
    caption: str | None = None,
    label: str | None = None,
    pm: str = r"$\pm$",
    std_size_cmd: str = r"\scriptsize",
    uppercase_tasks: bool = True,
    table_env: str = "table*",  # "table" or "table*"
    centering: bool = True,
):
    """Print a LaTeX table with best main metric mean±std per method×task.

    - Uses each task's main metric: `metrics_by_task[task][0]`.
    - Chooses the *best LR per method* from `best_lrs_dict[(task, rank)]`.
    - Aggregates across seeds and reports `mean ± std`.
    - `scale=100` makes values percentages.
    - Styles the best (bold) and optionally second-best (underline/italic) per task by mean.
    """
    method_to_pretty = method_to_pretty or {}
    rows = []
    for task in tasks:
        metric = metrics_by_task[task][0]
        df = task_df_dict[(task, rank)].copy()
        best_lr_df = best_lrs_dict[(task, rank)][["Method", "LR"]]
        df = df.merge(best_lr_df, on=["Method", "LR"], how="inner")
        sub = df[["Seed", "Method", "step", metric]].dropna()

        if best_over == "steps":
            idx = sub.groupby(["Seed", "Method"])[metric].idxmax()
            best_seed = sub.loc[idx]
        elif best_over == "fixed_step":
            mean_over_seeds = (
                sub.groupby(["Method", "step"], as_index=False)[metric].mean()
                .rename(columns={metric: "mean_metric"})
            )
            best_step = mean_over_seeds.loc[mean_over_seeds.groupby("Method")["mean_metric"].idxmax()][
                ["Method", "step"]
            ]
            best_seed = sub.merge(best_step, on=["Method", "step"], how="inner")
        else:
            raise ValueError(f"Unknown best_over: {best_over}")

        stats = (
            best_seed.groupby("Method")[metric]
            .agg([("mean", "mean"), ("std", "std"), ("n", "count")])
            .reset_index()
        )
        stats["Task"] = task
        rows.append(stats)

    all_stats = pd.concat(rows, ignore_index=True)
    mean = all_stats.pivot(index="Method", columns="Task", values="mean")
    std = all_stats.pivot(index="Method", columns="Task", values="std")

    ordered_methods = [m for m in method_to_pretty.keys() if m in mean.index] + [
        m for m in mean.index if m not in method_to_pretty
    ]
    mean = mean.loc[ordered_methods]
    std = std.loc[ordered_methods]

    formatted = pd.DataFrame(index=mean.index, columns=mean.columns, dtype=object)
    for task in tasks:
        for method in mean.index:
            formatted.loc[method, task] = _format_mean_std(
                mean.loc[method, task],
                std.loc[method, task],
                digits=digits,
                scale=scale,
                pm=pm,
                std_size_cmd=std_size_cmd,
            )

        ordered = mean[task].dropna().sort_values(ascending=False).index.tolist()
        if bold_best and len(ordered) >= 1:
            best_method = ordered[0]
            formatted.loc[best_method, task] = _latex_wrap(r"\textbf", formatted.loc[best_method, task])
        if highlight_second_best and len(ordered) >= 2:
            second_method = ordered[1]
            formatted.loc[second_method, task] = _latex_wrap(second_best_cmd, formatted.loc[second_method, task])

    formatted.index = [method_to_pretty.get(m, m) for m in formatted.index]
    formatted.index.name = "Method"

    if uppercase_tasks:
        formatted = formatted.rename(columns={t: t.upper() for t in formatted.columns})
    formatted.columns.name = None

    latex = formatted.to_latex(
        escape=False,
        index=True,
        column_format="l" + "c" * len(tasks),
        caption=caption,
        label=label,
    )

    # pandas emits a \begin{table} wrapper when caption/label are provided; switch to table* if requested.
    if table_env != "table":
        latex = latex.replace("\\begin{table}\n", f"\\begin{{{table_env}}}\n", 1)
        latex = latex.replace("\\end{table}\n", f"\\end{{{table_env}}}\n", 1)
        latex = latex.replace("\\begin{table}", f"\\begin{{{table_env}}}", 1)
        latex = latex.replace("\\end{table}", f"\\end{{{table_env}}}", 1)

    if centering:
        begin_with_newline = f"\\begin{{{table_env}}}\n"
        if begin_with_newline in latex:
            latex = latex.replace(begin_with_newline, begin_with_newline + "\\centering\n", 1)
        else:
            begin_inline = f"\\begin{{{table_env}}}"
            latex = latex.replace(begin_inline, begin_inline + "\n\\centering", 1)

    print(latex)
    return formatted, all_stats


_table_df, _raw_stats = print_latex_table_best_metrics(
    task_df_dict=task_df_dict,
    best_lrs_dict=best_lrs_dict,
    tasks=TASKS,
    rank=8,
    metrics_by_task=METRICS,
    method_to_pretty=method_to_pretty,
    digits=2,
    scale=100.0,
    bold_best=True,
    highlight_second_best=True,
    second_best_cmd=r"\underline",
    best_over="steps",
    caption=r"RoBERTa-base GLUE: best main metric (mean $\pm$ std), values in \%, over seeds.",
    label="tab:glue_roberta_best_metrics",
    pm=r"$\pm$",
    std_size_cmd=r"\scriptsize",
    uppercase_tasks=True,
    table_env="table*",
    centering=True,
 )